In [1]:
import numpy as np
import torch
# from public import config as cfg
# from public import utils
# from public import models

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import sys
import os
current_dir = "/home/dario/Projects/Federated_Learning/dful_exps/dful/cfl_drift"
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)
import public.config as cfg
import public.utils as utils
import public.models as models


In [4]:
def load_current_data(cur_round,train=True,descriptor_extraction=False):
    client_id = 1

    drifting_log = np.load(f'../data/cur_datasets/drifting_log.npy', allow_pickle=True).item()
    drifting_log = drifting_log[client_id]
    # load raw data
    if not cfg.training_drifting:
        cur_data = np.load(f'../data/cur_datasets/client_{client_id}.npy', allow_pickle=True).item()
    else:
        load_index = max([index for index in drifting_log if index <= cur_round], default=0)
        cur_data = np.load(f'../data/cur_datasets/client_{client_id}_round_{load_index}.npy', allow_pickle=True).item()
    
    cur_features = torch.tensor(cur_data['train_features'], dtype=torch.float32) if not cfg.training_drifting else torch.tensor(cur_data['features'], dtype=torch.float32)
    cur_labels = torch.tensor(cur_data['train_labels'], dtype=torch.int64) if not cfg.training_drifting else torch.tensor(cur_data['labels'], dtype=torch.int64)

    # cur_features = cur_data['train_features'] if not cfg.training_drifting else cur_data['features']
    cur_features = cur_features.unsqueeze(1) if utils.get_in_channels() == 1 else cur_features

    # cur_labels = cur_data['train_labels'] if not cfg.training_drifting else cur_data['labels']

    # Split the data into training and testing subsets
    train_features, val_features, train_labels, val_labels = train_test_split(
        cur_features, cur_labels, test_size=cfg.client_eval_ratio, random_state=cfg.random_seed
    )
    
    # reduce the number of samples 
    if cfg.n_samples_clients > 0:
        train_features = train_features[:cfg.n_samples_clients]
        train_labels = train_labels[:cfg.n_samples_clients]

    if train:
        train_dataset = models.CombinedDataset(train_features, train_labels, transform=None)
        return DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    else:
        val_dataset = models.CombinedDataset(val_features, val_labels, transform=None)
        # randomly sample the data for descriptor extraction
        if descriptor_extraction:
            if cfg.n_stochastic_sampling > 0:
                val_dataset_list = []
                for i in range(cfg.n_stochastic_sampling):
                    val_subset, _ = train_test_split(val_dataset, test_size=0.5, random_state= cfg.random_seed + cur_round**4 + i)
                    val_dataset_list.append(val_subset)
                val_dataset = torch.utils.data.ConcatDataset(val_dataset_list)
        return DataLoader(val_dataset, batch_size=cfg.test_batch_size, shuffle=False)

In [2]:
def load_current_data(cur_round, train=True, descriptor_extraction=False):
    client_id = 1

    drifting_log = np.load(f'../data/cur_datasets/drifting_log.npy', allow_pickle=True).item()
    drifting_log = drifting_log[client_id]
    
    # load raw data
    if not cfg.training_drifting:
        cur_data = np.load(f'../data/cur_datasets/client_{client_id}.npy', allow_pickle=True).item()
    else:
        load_index = max([index for index in drifting_log if index <= cur_round], default=0)
        # Load data for each client into a list
        cur_data_list = []
        for cid in range(1, 17):
            cur_data_list.append(
                np.load(f'../data/cur_datasets/client_{cid}_round_{load_index}.npy', allow_pickle=True).item()
            )
        # Merge dictionaries: for each key, concatenate values ensuring they are at least 1D.
        combined_data = {}
        for key in cur_data_list[0]:
            combined_data[key] = np.concatenate(
                [np.atleast_1d(client_data[key]) for client_data in cur_data_list], axis=0
            )
        cur_data = combined_data

    # Extract features and labels according to drifting mode
    if not cfg.training_drifting:
        cur_features = torch.tensor(cur_data['train_features'], dtype=torch.float32)
        cur_labels = torch.tensor(cur_data['train_labels'], dtype=torch.int64)
    else:
        cur_features = torch.tensor(cur_data['features'], dtype=torch.float32)
        cur_labels = torch.tensor(cur_data['labels'], dtype=torch.int64)

    # Adjust features shape if necessary
    # cur_features = cur_features.unsqueeze(1) if utils.get_in_channels() == 1 else cur_features

    # Split the data into training and validation subsets
    train_features, val_features, train_labels, val_labels = train_test_split(
        cur_features, cur_labels, test_size=cfg.client_eval_ratio, random_state=cfg.random_seed
    )
    
    # Optionally reduce the number of samples 
    if cfg.n_samples_clients > 0:
        train_features = train_features[:cfg.n_samples_clients]
        train_labels = train_labels[:cfg.n_samples_clients]

    if train:
        train_dataset = models.CombinedDataset(train_features, train_labels, transform=None)
        return DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    else:
        val_dataset = models.CombinedDataset(val_features, val_labels, transform=None)
        # For descriptor extraction, randomly sample the data
        if descriptor_extraction and cfg.n_stochastic_sampling > 0:
            val_dataset_list = []
            for i in range(cfg.n_stochastic_sampling):
                val_subset, _ = train_test_split(
                    val_dataset, test_size=0.5, random_state=cfg.random_seed + cur_round**4 + i
                )
                val_dataset_list.append(val_subset)
            val_dataset = torch.utils.data.ConcatDataset(val_dataset_list)
        return DataLoader(val_dataset, batch_size=cfg.test_batch_size, shuffle=False)

In [3]:
train_loader = load_current_data(0, train=True)

In [6]:
for x, y in train_loader:
    print(x.shape)
    print(y.shape)
    break

torch.Size([64, 1, 224, 224])
torch.Size([64, 14])


In [6]:
from torch import nn
import torch.nn.functional as F
# simple train function
def simple_train(model, device, train_loader, optimizer, epoch, client_id=None):
    model.train()
    loss_list = []
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        if cfg.dataset_name == "CheXpert":
            # For multi-label classification, ensure target is float
            loss = F.binary_cross_entropy_with_logits(output, target.float())
        else:
            # For multi-class classification (single label per sample)
            loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        # if batch_idx % 10 == 0:
        #     print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
        #           f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')
        loss_list.append(loss.item())
    print(f'Client: {client_id} - Train Epoch: {epoch} \tLoss: {sum(loss_list)/len(loss_list):.6f}')

model = models.ResNet9(in_channels=1, num_classes=14, input_size=(64,64)).to("cuda:2")

for epoch in range(3):
    simple_train(model=model,
                        device="cuda:2",
                        train_loader=train_loader, 
                        optimizer=torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum),
                        epoch=epoch,
                        client_id=1)

Client: 1 - Train Epoch: 0 	Loss: 0.555347
Client: 1 - Train Epoch: 1 	Loss: 0.389260
Client: 1 - Train Epoch: 2 	Loss: 0.317499


In [7]:
from sklearn.metrics import roc_auc_score
def simple_test(model, device, test_loader, dataset_name="default"):
    model.eval()
    test_loss = 0.0

    # For CheXpert (multi-label)
    if cfg.dataset_name == "CheXpert":
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                # Use BCE with logits for multi-label loss
                loss = F.binary_cross_entropy_with_logits(output, target.float(), reduction='sum')
                test_loss += loss.item()
                # Collect predictions and targets
                all_targets.append(target.cpu().numpy())
                all_preds.append(torch.sigmoid(output).cpu().numpy())
        # Concatenate results over batches
        all_targets = np.concatenate(all_targets, axis=0)
        all_preds = np.concatenate(all_preds, axis=0)
        
        # Compute macro-average AUROC over labels
        auc = roc_auc_score(all_targets, all_preds, average='macro')

        test_loss /= len(test_loader.dataset)
        return test_loss, auc

    # For multi-class datasets
    else:
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = F.cross_entropy(output, target, reduction='sum')
                test_loss += loss.item()
                pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
                correct += pred.eq(target.view_as(pred)).sum().item()
                total += target.size(0)
        test_loss /= len(test_loader.dataset)
        accuracy = correct / total
        return test_loss, accuracy

test_loader = load_current_data(0, train=False)
simple_test(model=model,
                    device="cuda:2",
                    test_loader=test_loader, 
                    dataset_name="default")

(4.747601318359375, 0.8584071105187527)

In [19]:
def get_in_channels():
    for file_name in ['../data/cur_datasets/client_1.npy', '../data/cur_datasets/client_1_round_-1.npy']:
        if os.path.exists(file_name):
            cur_data = np.load(file_name, allow_pickle=True).item()
            break
    cur_features = cur_data['train_features'] if not cfg.training_drifting else cur_data['features']
    print(f"cur_features shape: {cur_features.shape}")

    return 3 if len(cur_features.shape) == 4 else 1

print(get_in_channels())

cur_features shape: (50, 1, 224, 224)
3


In [24]:
from sklearn.metrics import roc_auc_score
def simple_test(model, device, test_loader, dataset_name="default"):
    model.eval()
    test_loss = 0.0

    # For CheXpert (multi-label)
    if cfg.dataset_name == "CheXpert":
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                # Use BCE with logits for multi-label loss
                loss = F.binary_cross_entropy_with_logits(output, target.float(), reduction='sum')
                test_loss += loss.item()
                # Collect predictions and targets
                all_targets.append(target.cpu().numpy())
                all_preds.append(torch.sigmoid(output).cpu().numpy())
        # Concatenate results over batches
        all_targets = np.concatenate(all_targets, axis=0)
        all_preds = np.concatenate(all_preds, axis=0)
        
        # Compute macro-average AUROC over labels
        auc = roc_auc_score(all_targets, all_preds, average='macro')

        test_loss /= len(test_loader.dataset)
        return test_loss, auc

    # For multi-class datasets
    else:
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = F.cross_entropy(output, target, reduction='sum')
                test_loss += loss.item()
                pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
                correct += pred.eq(target.view_as(pred)).sum().item()
                total += target.size(0)
        test_loss /= len(test_loader.dataset)
        accuracy = correct / total
        return test_loss, accuracy

test_loader = load_current_data(0, train=False)
simple_test(model=model,
                    device="cpu",
                    test_loader=test_loader, 
                    dataset_name="default")

(9.668990071614584, 0.5258436235596464)